<a href="https://colab.research.google.com/github/pvnskbs/GPU-learning/blob/main/vectorAddition.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vector Addition

In [1]:
!nvidia-smi
!pip install numba

Tue Jul 21 14:44:54 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   41C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [5]:
import numpy as np
from numba import cuda
import time

#---------------------------------
# GPU Kernel
#---------------------------------

@cuda.jit
def gpu_vector_add(A,B,C):

    i = cuda.grid(1)

    if i < C.size:
        C[i] = A[i] + B[i]

def cpu_vector_add(A,B):

    return A+B
#---------------------------------
# Inputs
#---------------------------------

N = 10

A = np.arange(N)

B = np.arange(N)*10

C = np.zeros(N)

start = time.perf_counter()

result = cpu_vector_add(A,B)

end = time.perf_counter()


print(f"CPU Time : {(end-start)*1000:.5f} ms")

C = np.zeros(N)
start1 = time.perf_counter()
#---------------------------------
# Copy to GPU
#---------------------------------

d_A = cuda.to_device(A)
d_B = cuda.to_device(B)
d_C = cuda.to_device(C)
cuda.synchronize()
end1 = time.perf_counter()

#---------------------------------
# Launch Configuration
#---------------------------------

threads_per_block = 256

blocks_per_grid = (N + threads_per_block -1)//threads_per_block


#---------------------------------
# Launch Kernel
#---------------------------------
start2 = time.perf_counter()
gpu_vector_add[blocks_per_grid, threads_per_block](d_A,d_B,d_C)
cuda.synchronize()
end2 = time.perf_counter()

#---------------------------------
# Copy Back
#---------------------------------

start3 = time.perf_counter()
result = d_C.copy_to_host()
cuda.synchronize()
end3 = time.perf_counter()

#---------------------------------
# Print
#---------------------------------

print(result)
print("Host --> Device")
print((end1-start1)*1000)

print()

print("Kernel Execution")
print((end2-start2)*1000)

print()

print("Device --> Host")
print((end3-start3)*1000)

CPU Time : 0.08387 ms
[ 0. 11. 22. 33. 44. 55. 66. 77. 88. 99.]
Host --> Device
1.7220439999618975

Kernel Execution
61.38399799999661

Device --> Host
0.22093600000516744


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 1 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
